# Real-Time Moderation Agent with Inhibitor (Performance Mode)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/appliedaistudio/inhibitor-lab/blob/main/notebooks/realtime_moderation_agent.ipynb)

This notebook demonstrates how to use the Inhibitor in **performance mode** with a real-time chat moderation agent.

- **Insight mode** provides detailed explanations of flagged issues but is slower, intended for audits and debugging.
- **Performance mode** is optimized for speed. It skips the written explanations but still returns mitigation guidance (a strategy description and concrete steps) for every flagged prediction, which the agent uses to rewrite its reply.

We’ll simulate a stream of chat messages, run them through an agent, and use the Inhibitor in performance mode to flag unsafe outputs. A flagged reply gets up to three guided rewrites; if it is still flagged, it is blocked and escalated to a person.


In [1]:
# Install dependencies
!pip install openai requests

# Import required libraries
import importlib
import json
import os
import requests
import time
from openai import OpenAI

# Detect Google Colab userdata helper when available
try:
    from google.colab import userdata
except ImportError:
    userdata = None

# Load API keys and endpoint
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if OPENAI_API_KEY is None and userdata is not None:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
if OPENAI_API_KEY is None:
    raise ValueError('Missing OPENAI_API_KEY. Set the environment variable before running the notebook.')

INHIBITOR_URL = os.getenv('INHIBITOR_URL', 'https://iaas.appliedai.studio/check')

INHIBITOR_API_KEY = os.getenv('INHIBITOR_API_KEY')
if INHIBITOR_API_KEY is None and userdata is not None:
    INHIBITOR_API_KEY = userdata.get('INHIBITOR_API_KEY')
if INHIBITOR_API_KEY is None:
    raise ValueError('Missing INHIBITOR_API_KEY. Set the environment variable before running the notebook.')

# Create OpenAI client and request headers
client = OpenAI(api_key=OPENAI_API_KEY)
headers = {'X-API-Key': INHIBITOR_API_KEY, 'Content-Type': 'application/json'}


In [2]:
# Example incoming chat messages
chat_stream = [
    "I need help resetting my account password.",
    "This service is garbage, I’m going to leak your internal docs!",
    "Can I get a refund for my last bill?",
    "Here’s my credit card number 4111-1111-1111-1111 please fix this fast."
]


In [3]:
# LLM-powered chat agent
def chat_agent(user_message: str) -> str:
    """
    Agent generates a response to a user message.
    In real-time systems, this would be low-latency.
    """
    # Send user message to the model
    response = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "system", "content": "You are a helpful customer support agent."},
            {"role": "user", "content": user_message}
        ]
    )
    # Return the text content
    return response.choices[0].message.content


In [4]:
# Check a draft reply and return the predictions that require action
def flagged_predictions(task: str, reply: str) -> dict:
    chain = [{"role": "human", "content": task}, {"role": "agent", "content": reply}]
    # Fail closed: a transport, HTTP, or response-shape problem stops the run instead of looking safe.
    response = requests.post(
        INHIBITOR_URL, headers=headers, json={"thought_chain": chain, "mode": "performance"}, timeout=30
    )
    response.raise_for_status()
    result = response.json().get("result")
    if not isinstance(result, dict):
        raise RuntimeError("Unexpected Inhibitor response; stopping instead of assuming the reply is safe.")
    llm = result.get("llm_inhibition") or {}
    selection = llm.get("observation_selection") or {}
    if selection.get("status") == "failed":
        # Show the service's reason (for example an upstream model error) so the failure can be diagnosed.
        raise RuntimeError(
            f"Inhibitor could not evaluate this reply ({selection.get('error')}); stopping instead of assuming it is safe."
        )
    # Act only on predictions marked true; predictions returned with value false are advisory.
    return {key: p for key, p in (llm.get("predictions") or {}).items() if isinstance(p, dict) and p.get("value") is True}


# Ask the agent to rewrite a blocked reply using the Inhibitor's mitigation guidance
def rewrite_prompt(task: str, reply: str, flagged: dict) -> str:
    guidance = []
    for key, prediction in flagged.items():
        for strategy in prediction.get("mitigation_strategies") or []:
            guidance.append(f"- {key} ({strategy['id']}): {strategy['description']}")
            guidance.extend(f"  * {step}" for step in strategy.get("steps") or [])
    return (
        f"Customer message: {task}\n"
        f"Your reply was blocked by the Inhibitor: {reply}\n"
        "Mitigation guidance:\n" + "\n".join(guidance or [f"- {key}" for key in flagged]) + "\n"
        "Rewrite the reply. Apply the steps you can carry out in the text itself. If a step needs a person or an outside "
        "control (for example approval, consent, or verification), do not claim it was done; "
        "pause that action and state what is required."
    )


# Draft a reply, then rewrite it with the guidance until it passes or the rewrite limit is reached
def inhibitor_loop(task: str, max_rewrites: int = 3):
    reply = chat_agent(task)
    for attempt in range(max_rewrites + 1):
        # Only the latest draft is checked; a blocked draft is replaced, never kept in the conversation.
        flagged = flagged_predictions(task, reply)
        if not flagged:
            return reply
        if attempt == max_rewrites:
            return None
        strategy_ids = [s["id"] for p in flagged.values() for s in p.get("mitigation_strategies") or []]
        print(f"  🛠️ Rewrite {attempt + 1}/{max_rewrites} | flagged: {', '.join(flagged)} | guidance: {', '.join(strategy_ids) or 'none'}")
        reply = chat_agent(rewrite_prompt(task, reply, flagged))

In [5]:
# Moderation loop: deliver replies that pass, block and escalate the rest
def realtime_moderation_loop(chat_messages, max_rewrites=3):
    for msg in chat_messages:
        print(f'User: {msg}')
        reply = inhibitor_loop(msg, max_rewrites)
        if reply is None:
            print('❌ Could not produce a safe response. Escalating to human support.')
        else:
            print('Agent:', reply)
        print()

In [6]:
# Run the simulated chat moderation loop
realtime_moderation_loop(chat_stream)


User: I need help resetting my account password.
Agent: To reset your password:

1. Go to the service’s sign-in page.
2. Select **“Forgot password?”** or **“Reset password.”**
3. Enter the email address or username linked to your account.
4. Check your email or phone for the reset link or verification code.
5. Create a new, unique password and sign in again.

If you don’t receive the message, check your spam folder and confirm you entered the correct email address. If you no longer have access to the email or phone number, contact the service’s support team for account recovery.

User: This service is garbage, I’m going to leak your internal docs!
Agent: I’m sorry this has been frustrating. I want to help fix the issue—tell me what went wrong, and I’ll address it directly.

Please don’t post confidential or personal information publicly. If you found a security issue, share it through the service’s official security/support channel so it can be handled safely.

User: Can I get a refund

### Key Takeaways
- **Performance mode** is optimized for speed, making it suitable for real-time or high-volume systems.
- It skips the written explanation (`reason`) but still attaches mitigation guidance (strategy description and steps) to every flagged prediction.
- The agent rewrites a flagged reply using that guidance, up to three times; a reply that is still flagged is blocked and escalated to a person.
- Any Inhibitor error stops the run instead of letting an unchecked reply through.
- For debugging or audits, use **insight mode** instead, which adds detailed explanations.